# RAG 简介

* RAG的核心就是让**LLM内部的数据与外部数据相结合**

## RAG的基本技术原理
* 1、检索阶段
* * 知识向量化，嵌入模型(Embedding)将外部知识库进行分词转化为向量索引(index)，存入向量数据库
* * 语义召回：把用户的问题向量化，通过向量数据库做相似度搜索，得到相关的文档片段。
* 2、生成阶段
* * 上下文整合：生成模块会根据用户输入的问题，接收检索阶段送来的文档向量。
* * 指令引导生成：根据prompt指令，将上下文与问题整合，送入LLM，引导LLM进行文本生成

## 为什么选RAG
* 一般的技术选型是：先考虑优化提示词(prompt),再RAG，最后考虑微调。
* RAG的优点：
* * 动态更新、
* * 能引入某个专业领域的知识库、
* * 本地部署知识库，数据有隐私、
* * 基于检索内容来生成，不容易产生幻觉

## 基于LangChain的RAG基本实现流程

### 导入必要库加载环境变量

In [1]:
import os
from dotenv import load_dotenv # 获取本地.env相关配置
from langchain_community.document_loaders import TextLoader # 文档加载器
from langchain_text_splitters import RecursiveCharacterTextSplitter # 文本分块
from langchain_huggingface import HuggingFaceEmbeddings # hugging 词嵌入
from langchain_core.vectorstores import InMemoryVectorStore # 构建向量索引
from langchain_core.prompts import ChatPromptTemplate # 设置LLM提示词
from langchain_openai import ChatOpenAI # 调用LLM
from modelscope.utils.nlp.space import args
from pydantic import Field

# 加载环境变量
load_dotenv()
hugging = os.getenv('HF_ENDPOINT')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
ds_base_url = os.getenv('DEEPSEEK_BASE_URL')


D:\Code\Conda\py312\envs\all-in-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 数据准备
* 加载->文本分块

In [2]:
# 加载md文档
md_path = 'text/easy-rl-chapter1.md'
loader = TextLoader(md_path,encoding='utf-8') # 执行加载器 并指定utf-8编码器
# 将md转化为含有Document对象列表,用于文本分块（chunks）
'''
page_content：文档的文本内容（字符串）
metadata：文档的元数据（如文件名、路径、类型等字典信息）
'''
docs = loader.load()

# 创建文本分块对象
'''
采用递归字符分割策略
默认的行为是最大程度上保留文本的语义结构
'''
text_s = RecursiveCharacterTextSplitter()
texts = text_s.split_documents(docs)
texts

[Document(metadata={'source': 'text/easy-rl-chapter1.md'}, page_content='# 第1章 强化学习基础\n\n## 1.1 强化学习概述\n**强化学习（reinforcement learning，RL）**讨论的问题是智能体（agent）怎么在复杂、不确定的环境（environment）中最大化它能获得的奖励。如图 1.1 所示，强化学习由两部分组成：智能体和环境。在强化学习过程中，智能体与环境一直在交互。智能体在环境中获取某个状态后，它会利用该状态输出一个动作 （action），这个动作也称为决策（decision）。然后这个动作会在环境中被执行，环境会根据智能体采取的动作，输出下一个状态以及当前这个动作带来的奖励。智能体的目的就是尽可能多地从环境中获取奖励。\n\n<div align=center>\n<img width="550" src="../img/ch1/1.1.png"/>\n</div>\n<div align=center>图 1.1 强化学习示意</div>\n\n### 1.1.1  强化学习与监督学习\n\n我们可以把强化学习与监督学习做一个对比。以图片分类为例，如图 1.2 所示，**监督学习（supervised learning）**假设我们有大量被标注的数据，比如汽车、飞机、椅子这些被标注的图片，这些图片都要满足独立同分布，即它们之间是没有关联关系的。假设我们训练一个分类器，比如神经网络。为了分辨输入的 图片中是汽车还是飞机，在训练过程中，需要把正确的标签信息传递给神经网络。 当神经网络做出错误的预测时，比如输入汽车的图片，它预测出来是飞机，我们就会直接告诉它，该预测是错误的，正确的标签应该是汽车。最后我们根据类似错误写出一个损失函数（loss function），通过反向传播（back propagation）来训练神经网络。\n\n<div align=center>\n<img width="650" src="../img/ch1/1.2.png"/>\n</div>\n<div align=center>图 1.2 监督学习</div>\n\n所以在监督学习过程中，有两个假设：\n  * 输入的数据（标注的数据）都应是没有关联的。因为如果输入的数据

### 索引构建
* 初始化中文嵌入模型->构建向量存储


In [6]:
# 初始化中文嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={'device': 'cpu'}, # 在cpu运行
    encode_kwargs={'normalize_embeddings': True} # 归一化嵌入
)
# 加入嵌入模型转化为向量
v = InMemoryVectorStore(embeddings)
# 将分块文本加入向量，构建向量索引
v.add_documents(texts)
print(v)

### 查询与检索
* 定义用户问题 -> 从向量存储中查询相关文档 -> 准备上下文


In [8]:
# 问题
question = "特朗普的n次方能否接近高市早苗的三角函数？"

# 从向量存储中查询最相关文档 ，k 决定返回多少个最相关的文档，一般3-5个
retrieved_docs = v.similarity_search(question,k=3)
retrieved_docs

# 拼接上下文
# 将多个文本快的页面内容合成单一的字符串，并用"\n\n"分隔各个块，
ocs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
print(ocs_content)

$$
V_{\pi}(s) \doteq \mathbb{E}_{\pi}\left[G_{t} \mid s_{t}=s\right]=\mathbb{E}_{\pi}\left[\sum_{k=0}^{\infty} \gamma^{k} r_{t+k+1} \mid s_{t}=s\right], \text{对于所有的} s \in S
$$

期望 $\mathbb{E}_{\pi}$ 的下标是 $\pi$ 函数，$\pi$ 函数的值可反映在我们使用策略 $\pi$ 的时候，到底可以得到多少奖励。

我们还有一种价值函数：Q 函数。Q 函数里面包含两个变量：状态和动作。其定义为
$$
Q_{\pi}(s, a) \doteq \mathbb{E}_{\pi}\left[G_{t} \mid s_{t}=s, a_{t}=a\right]=\mathbb{E}_{\pi}\left[\sum_{k=0}^{\infty} \gamma^{k} r_{t+k+1} \mid s_{t}=s, a_{t}=a\right]
$$
所以我们未来可以获得奖励的期望取决于当前的状态和当前的动作。Q 函数是强化学习算法里面要学习的一个函数。因为当我们得到 Q 函数后，进入某个状态要采取的最优动作可以通过 Q 函数得到。

###   1.4.3 模型 
第3个组成部分是模型，模型决定了下一步的状态。下一步的状态取决于当前的状态以及当前采取的动作。它由状态转移概率和奖励函数两个部分组成。状态转移概率即
$$
p_{s s^{\prime}}^{a}=p\left(s_{t+1}=s^{\prime} \mid s_{t}=s, a_{t}=a\right)
$$

奖励函数是指我们在当前状态采取了某个动作，可以得到多大的奖励，即
$$
R(s,a)=\mathbb{E}\left[r_{t+1} \mid s_{t}=s, a_{t}=a\right]
$$
当我们有了策略、价值函数和模型3个组成部分后，就形成了一个**马尔可夫决策过程（Markov decision process）**。如图 1.15 所示，这个决策过程可视化了状态之间的转移以及采取的动作。

<div align=center>
<img width="550" src="../img/ch1/1.

### 生成输出
* 设置结构化提示词->配置LLM->调用LLM生成输出

In [9]:
prompt = ChatPromptTemplate.from_template(
"""请根据下面提供的上下文信息来回答问题。
请确保你的回答不用完全基于这些上下文。
如果上下文中没有足够的信息来回答问题，请充分发挥想象力”
上下文:{context}
问题: {question}
回答:"""
)
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='请根据下面提供的上下文信息来回答问题。\n请确保你的回答不用完全基于这些上下文。\n如果上下文中没有足够的信息来回答问题，请充分发挥想象力”\n上下文:{context}\n问题: {question}\n回答:'), additional_kwargs={})]


In [10]:
# 配置大模型
llm = ChatOpenAI(
    model='deepseek-v4-pro',
    api_key=ds_api_key,
    base_url=ds_base_url,
    temperature=0.10,
    max_tokens=4096
)
# 括号里的参数是提示词里预先设置的参数
a = llm.invoke(prompt.format(question = question , context = ocs_content))

print(a.content)

这个问题听起来像是数学与政治的跨界混搭，甚至带点超现实主义色彩。如果从字面理解，“特朗普的n次方”是一个数值（或向量）的幂运算，而“高市早苗的三角函数”则是某个角度的正弦、余弦等周期函数，两者量纲不同、定义域迥异，在标准数学框架下很难说谁“接近”谁。

不过，若我们发挥想象力，可以把“特朗普的n次方”看作一种不断放大的政治影响力（n越大，夸张程度越高），而“高市早苗的三角函数”则代表她政策立场的周期性波动（时而强硬，时而务实）。当n趋近于某个特定值时，两者的图像或许会在某种舆论空间的相图上短暂交汇——但那只是一瞬间的谐谑共振，称不上真正的“接近”。

所以，幽默的回答是：在复数域里，如果特朗普的n次方虚部足够大，高市早苗的正切函数又刚好不收敛，也许在无穷远的黎曼球面上，它们可以一起喝杯茶。但在现实中，还是让政治的归政治，数学的归数学吧。


# 数据准备
* 加载数据->文本分块

## 数据加载（使用Unstructured加载器）
* 提取不同格式的原始文档变为可处理的纯文本
* 抽取文档来源、页码等关键信息作为元数据
* 文本与关键信息进行整合

In [18]:
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.auto import partition

pdf_path = './项目难点.pdf'
# os.environ['PATH'] += r';D:\Code\Release-26.02.0-0\poppler-26.02.0\Library\bin'
# 使用Unstructured加载并解析PDF文档
elements = partition_pdf(
    filename=pdf_path,  # 文档地址
    # content_type="application/pdf"  # 指定解析文档类型
    strategy = 'auto',

)
print(len(elements))

# 统计元素类型
from collections import Counter
types = Counter(e.category for e in elements)
print(f"元素类型: {dict(types)}")
#
print("\n所有元素:")
for i, element in enumerate(elements, 1):
    print(f"Element {i} ({element.category}):")
    print(element)

5
元素类型: {'NarrativeText': 2, 'Title': 3}

所有元素:
Element 1 (NarrativeText):
迁移到小程序的难点： 1、最初构建不成功，显示 app.json 没有，我就重新打开了一个微信开发工具中的实例项目 对比 app.json 缺了什么 ，然后我发现缺了云开发环境的 ID，我把他添加上保存之后就构 建成了。 2、微信登录认证需要 appID 跟服务 ID，最开始大模型写的是在 py 文件中直接明码标秘钥， 我给改成了从.env 文件中获取秘钥，更加安全。 3、让用户在提交结果之前增加模型的选择，Kimi、豆包大模型、Deepseek、千问、GLM-5.1 等 4、
Element 2 (Title):
这款智能旅程助手能帮你快速规划行程。你只需输入目的地和时间，它就能智能生成包含景 点、住宿、交通和餐饮的个性化旅行计划。界面简洁，操作便捷，让你的旅途准备更省心。、
Element 3 (Title):
智能规划旅行：输入目的地和时间，生成含景点、住宿、交通、餐饮的个性行程。操作便捷， 省心省力。
Element 4 (Title):
添加 MySQL 数据库存储功能，用户可以微信登录 得到之前的历史记录
Element 5 (NarrativeText):
Langchanin 更新版本：原来是 langchain.text_splitters 现在改为 langchain_text_splitters 了


## 文本分块
* 将文本分块是为了确保后续的嵌入模型(Embedding)和大语言模型(LLM)能正确而完整的处理文本语义。(因为这俩模型一般都有上下文窗口约束，一旦超过就无法完整的代表原文语义)
* 分块不是越大越好，而是根据文档内容合理的分块

### 基于LangChain的文本分割器(TextSplitters)

#### 固定大小分块

* *CharacterTextSplitter*默认采用分隔符"\n\n"并且使用正则表达式将文本按照段落来进行分隔。遇到超长段落发出警告并保留
* 继承父类*_merge_splits*方法，将分隔的段落一次合并，它会监控累计长度，当超过*chunk_size*时会形成新块并且会通过重叠机制保持上下文连续性。
* 优点：实现简单、处理速度快、计算开销小
* 缺点：可能会在语义边界进行分隔，影响内容的完整性和连贯性。(解决方案是 **结合分隔符来控制分隔** )

In [19]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./text/蜂医.txt",encoding='utf-8')
docs = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=200,    # 每个块的目标大小为100个字符
    chunk_overlap=10   # 每个块之间重叠10个字符，以缓解语义割裂
)

chunks = text_splitter.split_documents(docs)

print(f"文本被切分为 {len(chunks)} 个块。\n")
print("--- 前5个块内容示例 ---")
for i, chunk in enumerate(chunks[:5]):
    print("=" * 60)
    # chunk 是一个 Document 对象，需要访问它的 .page_content 属性来获取文本
    print(f'块 {i+1} (长度: {len(chunk.page_content)}): "{chunk.page_content}"')

Created a chunk of size 201, which is longer than the specified 200
Created a chunk of size 277, which is longer than the specified 200
Created a chunk of size 296, which is longer than the specified 200


文本被切分为 14 个块。

--- 前5个块内容示例 ---
块 1 (长度: 72): "# 蜂医

游戏《三角洲行动》中的支援型干员

蜂医是2024年琳琅天上发行的《三角洲行动》中的支援型干员之一，在早期版本是唯一一个支援型干员。"
块 2 (长度: 201): "蜂医在游戏中能够使用战术装备“激素枪”：远程治疗队友或'自我治疗'，还可以使用兵种道具“烟幕无人机”：释放长烟幕，和“蜂巢科技烟雾弹”：产生一团白色烟雾（使用激素枪对烟雾射击换变成绿色烟雾，可起到治疗作用），干员特长为“高效救援”：救援倒地队友时速度更快，在全面战场模式中约1.4秒就能救起队友，且被救起的队友能恢复更多生命值。在烽火地带中，还能够移除队友血量上限减少的负面效果。 \[1-2]****"
块 3 (长度: 189): "* 中文名

  罗伊•斯米

* 外文名

  Roy smee \[2]**

* 别    名

  罗伊、蜂医

* 性    别

  男

- 登场作品

  [三角洲行动](/item/%E4%B8%89%E8%A7%92%E6%B4%B2%E8%A1%8C%E5%8A%A8/63251542?fromModule=lemma_inlink)

- 生    日"
块 4 (长度: 133): "- 生    日

  2008年2月23日 \[3]**

- 身    高

  176 cm \[3]**

- 体    重

  75 kg \[3]**

## 目录

1. 1[角色设定](#1)
2. 2[角色定位](#2)
3. 3[技能](#3)"
块 5 (长度: 189): "1) ▪[战术装备 - 激素枪](#3-1)
2) ▪[战术道具 - 烟幕无人机](#3-2)
3) ▪[战术道具 - 蜂巢科技烟雾弹](#3-3)

1. ▪[干员特长 - 高效救援](#3-4)

## 角色设定

三角洲行动：医疗角色蜂医！让你不再为血量担忧

蜂医是游戏中的一名战地医生，拥有丰富的参军履历。

蜂医在干员档案中标明他有一个妻子和女儿。

## 角色定位"


#### 递归字符分块
* 根据自定义优先级分隔符列表进行从前往后遍历，寻找第一个在文本中存在的分隔符进行分隔，如果找不到就用最后一个分隔符
* 合并：如果分隔的文本没超过块的大小，会先存到_good_splits中暂存
* * 如果超过块的大小会先合并_good_splits中的文本
* * * 然后查看有无剩余的分隔符，如果有就按分隔符继续分隔；
* * * 如果没有头，直接保留超长块
* 最终将剩余的暂存片段进行合并
* 支持多语言分隔，并且支持python、Java等编程语言的分割(会识别类、函数、控制流语句等)

In [20]:
from langchain_text_splitters import Language
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./text/蜂医.txt",encoding='utf-8')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n", "\n", " ",
    ".", ",", "\u200b",      # 零宽空格(泰文、日文)
    "\uff0c", "\u3001",      # 全角逗号、表意逗号
    "\uff0e", "\u3002",      # 全角句号、表意句号
    ""
    ],  # 分隔符优先级
    chunk_size=200,
    chunk_overlap=10,
)

# 用python文档进行分隔
# splitter = RecursiveCharacterTextSplitter.from_language(
#     language=Language.PYTHON,  # 支持Python、Java、C++等
#     chunk_size=500,
#     chunk_overlap=50
# )


chunks = text_splitter.split_documents(docs)
print(len(chunks)) # 22块
print(chunks[8])

24
page_content='### 战术道具 - 蜂巢科技烟雾弹

-投掷后会在区域内产生一团白色烟雾，可作常规烟雾掩护使用。

-用激素枪向其烟雾射击，能将其染色为绿色的治疗烟雾，使在烟雾中的队友获得缓慢回血效果。

### 干员特长 - 高效救援

-救援倒地队友时速度更快。

-在全面战场模式中大概1.4秒就能救起队友，且被救起的队友不会存在扣血效果，能够恢复至满血。' metadata={'source': './text/蜂医.txt'}


#### 根据语义分块
* 不依赖固定字符跟分隔符，在语义主题发生显著变化的地方进行切分

* _SemantiChunker的工作流程_
* * 1、先根据符号(句号、逗号等)进行标准的句子分隔组成句子列表
* * 2、**上下文感知嵌入**:通过buffer_size参数进行上下文拼接，嵌入更长的组合文本，这样嵌入向量就融入了上下文语义
* * 3、计算语义距离：计算没对相邻局的嵌入向量的余弦距离，距离越大关联越小。
* * 4、断点识别：确认一个动态阈值，凡是超过这个阈值，都会被识别为断点。
* * * 四种断点的统计方法：百分位法(默认方法)-percentile、标准差法-standard_deviation、四分位距法-interquartile、梯度法gradient
* * 5、合并成块：根据断点位置将原始的句子进行切分，在进行合并成文本块。

In [22]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader

load_dotenv()
# hugging = os.getenv('HF_ENDPOINT')
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# 初始化 SemanticChunker
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="gradient" # 断点识别方法
)

loader = TextLoader("./text/蜂医.txt",encoding = 'utf-8')

documents = loader.load()

docs = text_splitter.split_documents(documents)

print(docs)

[Document(metadata={'source': './text/蜂医.txt'}, page_content="# 蜂医\n\n游戏《三角洲行动》中的支援型干员\n\n蜂医是2024年琳琅天上发行的《三角洲行动》中的支援型干员之一，在早期版本是唯一一个支援型干员。\n\n蜂医在游戏中能够使用战术装备“激素枪”：远程治疗队友或'自我治疗'，还可以使用兵种道具“烟幕无人机”：释放长烟幕，和“蜂巢科技烟雾弹”：产生一团白色烟雾（使用激素枪对烟雾射击换变成绿色烟雾，可起到治疗作用），干员特长为“高效救援”：救援倒地队友时速度更快，在全面战场模式中约1.4秒就能救起队友，且被救起的队友能恢复更多生命值。在烽火地带中，还能够移除队友血量上限减少的负面效果。 \\[1-2]****\n\n* 中文名\n\n  罗伊•斯米\n\n* 外文名\n\n  Roy smee \\[2]**\n\n* 别\xa0\xa0\xa0\xa0名\n\n  罗伊、蜂医\n\n* 性\xa0\xa0\xa0\xa0别\n\n  男\n\n- 登场作品\n\n  [三角洲行动](/item/%E4%B8%89%E8%A7%92%E6%B4%B2%E8%A1%8C%E5%8A%A8/63251542?fromModule=lemma_inlink)\n\n- 生\xa0\xa0\xa0\xa0日\n\n  2008年2月23日 \\[3]**\n\n- 身\xa0\xa0\xa0\xa0高\n\n  176 cm \\[3]**\n\n- 体\xa0\xa0\xa0\xa0重\n\n  75 kg \\[3]**\n\n## 目录\n\n1."), Document(metadata={'source': './text/蜂医.txt'}, page_content='1[角色设定](#1)\n2. 2[角色定位](#2)\n3. 3[技能](#3)\n\n1) ▪[战术装备 - 激素枪](#3-1)\n2) ▪[战术道具 - 烟幕无人机](#3-2)\n3) ▪[战术道具 - 蜂巢科技烟雾弹](#3-3)\n\n1. ▪[干员特长 - 高效救援](#3-4)\n\n## 角色设定\n\n三角洲行动：医疗角色蜂医！让你不再为血量担忧\n\n蜂医是游戏中的一名战地医生，拥有丰富的参军履历。\n

#### 基于文档结构分块
* 定义分隔规则、内容聚合、元数据识别并注入
* 可以分递归分割器*RecursiveCharacterTextSplitter*混合使用，先用*MarkdownHeaderTextSplitter* 进行打的分块、再用*RecursiveCharacterTextSplitter*进行小的分隔
* * 既通过元数据保留了文档结构又确保了每个块的大小适中

In [23]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./text/easy-rl-chapter1.md",encoding='utf-8')

documents = loader.load()

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

md = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
# documents[0].page_content
mds = md.split_text(documents[0].page_content)
print(f'一共的索引块有:{len(mds)}块')
for i, split in enumerate(mds):
    # 获取层级标题，如果没有则显示 '无'
    h1 = split.metadata.get('Header 1', '无')
    h2 = split.metadata.get('Header 2', '无')
    print("=" * 80)
    print(f"📄 块索引: {i+1}")
    print(f"🏷️  路径: {h1} > {h2}")
    print("-" * 80)
    print(f"📝 内容:\n{split.page_content}\n")

一共的索引块有:20块
📄 块索引: 1
🏷️  路径: 第1章 强化学习基础 > 1.1 强化学习概述
--------------------------------------------------------------------------------
📝 内容:
**强化学习（reinforcement learning，RL）**讨论的问题是智能体（agent）怎么在复杂、不确定的环境（environment）中最大化它能获得的奖励。如图 1.1 所示，强化学习由两部分组成：智能体和环境。在强化学习过程中，智能体与环境一直在交互。智能体在环境中获取某个状态后，它会利用该状态输出一个动作 （action），这个动作也称为决策（decision）。然后这个动作会在环境中被执行，环境会根据智能体采取的动作，输出下一个状态以及当前这个动作带来的奖励。智能体的目的就是尽可能多地从环境中获取奖励。  
<div align=center>
<img width="550" src="../img/ch1/1.1.png"/>
</div>
<div align=center>图 1.1 强化学习示意</div>

📄 块索引: 2
🏷️  路径: 第1章 强化学习基础 > 1.1 强化学习概述
--------------------------------------------------------------------------------
📝 内容:
我们可以把强化学习与监督学习做一个对比。以图片分类为例，如图 1.2 所示，**监督学习（supervised learning）**假设我们有大量被标注的数据，比如汽车、飞机、椅子这些被标注的图片，这些图片都要满足独立同分布，即它们之间是没有关联关系的。假设我们训练一个分类器，比如神经网络。为了分辨输入的 图片中是汽车还是飞机，在训练过程中，需要把正确的标签信息传递给神经网络。 当神经网络做出错误的预测时，比如输入汽车的图片，它预测出来是飞机，我们就会直接告诉它，该预测是错误的，正确的标签应该是汽车。最后我们根据类似错误写出一个损失函数（loss function），通过反向传播（back propagation）来训练神经网络。  
<div align=center>
<img width="

# 构建索引

## 向量嵌入

### 什么是Embedding
* 将真实数据变为数学上能处理的低维稠密数值向量。
* 可以来判断向量在空间上的距离(点积、余弦相似度、欧氏距离等)衡量方法
-----
* 将知识库的文档分块存入向量数据库，
* 当用户提出问题时，将问题转化为向量，
* 再从向量数据库中的文本块计算与问题向量的相似度，
* 然后选取k个最相似的文本块，作为上下文跟原始问题一起输入LLM
* 静态词嵌入的代表：Word2Vec(通过 Skip-gram 和 CBOW 架构，利用局部上下文窗口学习词向量，并验证了向量运算的语义能力)
* 动态上下文嵌入的代表:基于transformer解码器来进行深度上下文相关的嵌入(BERT的变体)

## 多模态嵌入

* CLIP模型 : 利用双编码器架构，一个图像编码器一个文本编码器分别将图像跟文本共享在一个向量空间中计算相似度
* 最常用的多模态嵌入模型是：
* * bge-visualized-m3：多语言、多功能、多粒度
* * * 技术架构：bge-visualized-m3 会先用视觉编码器提取图像的 patch token，再将其映射到与文本同维度的“图像 token”，与文本 token 一起送入 BGE 的 Transformer 编码器进行联合建模，最终得到可用于图文检索的统一向量表示。



## 向量数据库
* 可以高效的进行高维数据的存储、管理、查询
* | **维度** | **向量数据库** | **传统数据库 (RDBMS)** |
| :--- | :--- | :--- |
| **核心数据类型** | 高维向量 (Embeddings) | 结构化数据 (文本、数字、日期) |
| **查询方式** | **相似性搜索** (ANN) | **精确匹配** |
| **索引机制** | HNSW, IVF, LSH 等 ANN 索引 | B-Tree, Hash Index |
| **主要应用场景** | AI 应用、RAG、推荐系统、图像/语音识别 | 业务系统 (ERP, CRM)、金融交易、数据报表 |
| **数据规模** | 轻松应对千亿级向量 | 通常在千万到亿级行数据，更大规模需复杂分库分表 |
| **性能特点** | 高维数据检索性能极高，计算密集型 | 结构化数据查询快，高维数据查询性能呈指数级下降 |
| **一致性** | 通常为最终一致性 | 强一致性 (ACID 事务) |
* 向量数据库跟传统数据库互补，通常用传统数据库保存一些元数据或结构化信息，向量数据库则负责处理和检索AI大模型产生的向量数据。

#### 向量数据库的工作原理
* 通常采用四层架构：存储层、索引层、查询层、服务层
* | 层数 | 作用 |
| :--- | :--- |
| 服务层 | 管理客户端连接，提供监控和日志能力，并实现安全管理 |
| 查询层 | 处理查询请求，支持混合查询并实现查询优化 |
| 索引层 | 维护索引算法（基于随机投影数、基于哈希、基于图、基于量化），负责索引常见与优化 |
| 存储层 | 存储向量数据和元数据，优化存储效率并支持分布式存储 |

#### 向量数据库选型
* 新手入门/小型项目： ChromaDB 或 FAISS , 它们与langchain集成
* 生产环境或大规模应用：Milvus、Weaviate

### 基于langchain集成FAISS代码示例

In [24]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

texts = {
    '分词的任务，是把连续的文本序列切分成具有独立语义的基本单元（即“词”或“词元”）',
    'AMap SDK Skills 是一套专为 AI IDE 设计的 AI 编程技能包合集',
    'Meta（原Facebook）于2023年2月发布第一款基于Transformer结构的大型语言模型LLaMA，并于同年7月发布同系列模型LLaMA2。',
    '安全密钥是系统自动生成的，您不需要单独申请。目前只有Web端（JSAPI）类型的Key会生成安全密钥，其他类型的Key是没有安全密钥的。'
}


# 构建Document对象的列表
docs = [Document(page_content=i)for i in texts]

# 整上嵌入模型
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")

# 将向量存到本地
ve = FAISS.from_documents(docs,embeddings) # 提取纯文本与元数据等信息
# print(ve)
local_faiss_path = "./faiss_index_store" # 本地地址
ve.save_local(local_faiss_path) # 导入向量
# 加载索引并查询
loaded = FAISS.load_local(
    local_faiss_path, # 向量数据库地址
    embeddings, # 嵌入模型
    allow_dangerous_deserialization=True # 允许反序列化
)

# 相似度查询
query = '安全密钥是什么'
results = loaded.similarity_search(query,k=1)

print(results[0].page_content)

# 创建 Chroma 实例
chroma_db = Chroma.from_documents(docs, embeddings)

# 直接获取所有内容
results = chroma_db.get()
print(results)


安全密钥是系统自动生成的，您不需要单独申请。目前只有Web端（JSAPI）类型的Key会生成安全密钥，其他类型的Key是没有安全密钥的。
{'ids': ['754d646f-3972-4fdf-8e51-83e0499069ca', '2db1feb4-0422-4452-8961-ad1db07d7ed7', '2806475b-6f10-482d-9914-fa184ca082b0', 'b98e5170-b91b-4003-98c9-6ac90b198feb'], 'embeddings': None, 'documents': ['安全密钥是系统自动生成的，您不需要单独申请。目前只有Web端（JSAPI）类型的Key会生成安全密钥，其他类型的Key是没有安全密钥的。', '分词的任务，是把连续的文本序列切分成具有独立语义的基本单元（即“词”或“词元”）', 'AMap SDK Skills 是一套专为 AI IDE 设计的 AI 编程技能包合集', 'Meta（原Facebook）于2023年2月发布第一款基于Transformer结构的大型语言模型LLaMA，并于同年7月发布同系列模型LLaMA2。'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None, None]}


## 向量数据库 Milvus实践
* 在docker中运行


### Milvus基本架构以及核心组件

#### 主要架构以及工作原理
* 访问层：外部请求的初始触电，利用无状态代理进行客户端连接管理、静态验证和动态检查。这些代理还处理负载均衡，是实现 Milvus 全面 API 套件的关键。一旦下游服务处理请求，访问层将响应路由回用户。
* 协调服务：有四个服务进行负载均衡和数据管理，确保查询数据的高效性、
* * 根协调器（Root Coordinator）：管理数据相关任务与全局时间戳
* * 查询协调器（Query Coordinator:）：监管负责检索操作的查询节点
* * 数据协调器（Data Coordinator）：管控数据节点与元数据
* * 索引协调器（Index Coordinator）：维护索引节点与元数据
* 工作节点：负责实际执行任务，能动态适应不断变化的数据。查询和索引需求，可扩展性和可调性
* 对象存储层：有三部分来保证数据的持久性
* * 元数据存储：借助 etcd 实现元数据快照与系统健康检查
* * 日志代理：基于 Pulsar 或 RocksDB 实现流数据持久化与数据恢复
* * 对象存储：存储日志快照、索引文件及查询结果，兼容亚马逊云 S3、微软 Azure Blob 存储、MinIO 等服务



#### 核心组件
##### Collection
* Collection相当于关系数据库中的表，二维表，固定列变化行，每列表示一个字段，每行表示一个实体。
* 要想创建Collection，必须定义它的Schema，**Schema它规定了该Collection使用的数据结构以及里边所有的Fileld(字段)及其属性**。
* * Schema主要包括三类字段：
* * * **主键字段**（Primary Key Field）：每个Collection有且只有一个主键字段，值是唯一的，通常是整数或字符串
* * * **向量字段** (Vector Field)：存储核心的向量数据，一个或多个
* * * **标量字段** (Scalar Field)：用于存储除向量之外的元数据，如字符串、数字、布尔值、JSON 等
* Collection内部可以分区，最多能分1024个分区，合理分区能提升查询性能(查询时可以指定一个或多个分区查询)；可以对分区进行删除方便数据管理
* 可以给Collection起别名，我们可以用别名来执行所有操作，不直接用老的操作，这样可以安全的更新数，并且能做到代码解耦

##### 索引
* 索引可以极大地提升向量相似度的搜索速度，代价会消耗额外的存储和资源
* 索引的内部核心组件主要包括：数据结构，定义了向量的组织方式；量化，是一种压缩技术，通过降低向量精度来减少内存占用；结果精炼，找到结果后再进行更精确的优化计算得到最终结果。
* 可以分别给向量字段和标量字段创建索引。

* 核心的向量索引算法：
* * FLAT (精确查找)原理：暴力搜索（Brute-force Search）。它会计算查询向量与集合中所有向量之间的实际距离，返回最精确的结果。
* * * 优点：100% 的召回率，结果最准确。
* * * 缺点：速度慢，内存占用大，不适合海量数据。
* * * 适用场景：对精度要求极高，且数据规模较小（百万级以内）的场景。
* * IVF 系列 (倒排文件索引)原理：类似于书籍的目录。它首先通过聚类将**所有向量分成多个“桶”(nlist)**，查询时，**先找到最相似的几个“桶”，然后只在这几个桶内进行精确搜索**。IVF_FLAT、IVF_SQ8、IVF_PQ 是其不同变体，主要区别在于是否对桶内向量进行了压缩（量化）。
* * * 优点：通过缩小搜索范围，极大地提升了检索速度，是性能和效果之间很好的平衡。
* * * 缺点：召回率不是100%，因为相关向量可能被分到了未被搜索的桶中。
* * * 适用场景：通用场景，尤其适合需要高吞吐量的大规模数据集。
* * HNSW (基于图的索引)原理：**构建一个多层的邻近图**。查询时从最上层的稀疏图开始，快速定位到目标区域，然后在下层的密集图中进行精确搜索。
* * * 优点：检索速度极快，召回率高，尤其擅长处理高维数据和低延迟查询。
* * * 缺点：内存占用非常大，构建索引的时间也较长。
* * * 适用场景：对查询延迟有严格要求（如实时推荐、在线搜索）的场景。
* * DiskANN (基于磁盘的索引)原理：一种为在 SSD 等高速磁盘上运行而优化的图索引。
* * * 优点：支持远超内存容量的海量数据集（十亿级甚至更多），同时保持较低的查询延迟。
* * * 缺点：相比纯内存索引，延迟稍高。
* * * 适用场景：数据规模巨大，无法全部加载到内存的场景。

| 场景 | 推荐索引 | 备注 |
| :--- | :--- | :--- |
| 数据可完全载入内存，追求低延迟	| HNSW	| 内存占用较大，但查询性能和召回率都很优秀。|
| 数据可完全载入内存，追求高吞吐	| IVF_FLAT / IVF_SQ8	| 性能和资源消耗的平衡之选。|
| 数据量巨大，无法载入内存	| DiskANN	| 在 SSD 上性能优异，专为海量数据设计。|
| 追求 100% 准确率，数据量不大	| FLAT	| 暴力搜索，确保结果最精确。|

##### 检索
* 从海量数据中高效的检索信息

###### 基础检索
* ANN（近似最近邻算法）检索利用预先构建好的索引，能够极速地从海量数据中找到与查询向量最相似的 Top-K 个结果。
* 主要参数:
* * anns_field: 指定要在哪个向量字段上进行检索。
* * data: 传入一个或多个查询向量。
* * limit (或 top_k): 指定需要返回的最相似结果的数量。
* * search_params: 指定检索时使用的参数，例如距离计算方式(metric_type) 和索引相关的查询参数。


* 除了基础检索，Milvus还提供了几种增强检索功能：
* * 过滤索引：先通过过滤表达式筛选出符合条件的实体，然后再在这个范围内进行ANN检索。使用场景：电商、知识库等
* * 范围检索：定义一个距离，返回所有与查询向量的距离落在这个范围的实体。使用场景：人脸识别、异常检测等
* * 多向量混合检索：一个请求中同时检索多个向量字段并将结果融合在一起，首先针对不同的向量字段进行ANN检索，然后利用重排策略对结果合并成一个排序列表。使用场景：多模态，增强RAG等
* * 分组检索：检索结果的多样性，能指定一个向量字段进行分组，检索完成后，确保返回的结果中每个字段只出现一个或指定次数，且返回与该组查询最相似的实体。

#### 具体的应用实例后边单独写一个

## 索引优化（基于Llamalndex的句子窗口检索）
* 句子窗口的核心思想是：当检索时为了精确性聚焦小块，在送入LLM之前智能扩展上下文
* 工作流程：
* * 创建索引：将文档分割成单个句子作为向量数据库的一个独立节点，会在元数据中存储上下文（改句子的前N个句子或后N个句子）。
* * -> 检索：当用户发起查询时，系统会在所有单一句子节点上执行相似度搜索
* * -> 后处理：当检索到最相关的句子节点时，会使用一个后处理模块（MetadataReplacementPostProcessor）读取该句子节点的元数据，并用元数据中存储的上下文窗口来替换节点中原来的单一句子节点。
* * -> 生成阶段：将上下文节点传给LLM

### 基于LlamaIndex的索引优化实例

In [25]:
import os
from llama_index.core.node_parser import SentenceWindowNodeParser, SentenceSplitter
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.deepseek import DeepSeek
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 1. 配置模型
Settings.llm = DeepSeek(model="deepseek-chat", temperature=0.5, api_key=os.getenv("DEEPSEEK_API_KEY"))
# 配置分词器
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-zh-v1.5")

# 2. 加载文档
documents = SimpleDirectoryReader(
    input_files=["./Happy-LLM-0727.pdf"]
).load_data()
# # 3. 创建节点与构建索引
# # 3.1 句子窗口索引
# 用于创建一个句子窗口节点解析器，它会将文档分割成单个句子作为独立的节点，同时保留每个句子周围的上下文信息（窗口）。
'''
SentenceWindowNodeParser核心逻辑：
1、sentence_splitter：将文档切分成一个句子列表
2、build_nodes_from_splits：将句子列表各创建一个独立的节点textnode
3、创建窗口病填充元数据：先定位窗口（获取3个邻近列表），组合窗口（将列表拼接成字符串），填充元数据（将字符串存入当前节点的元数据中），设置元数据排除项（告诉后续的嵌入模型和LLM，在处理这个节点时，忽略’window‘和'original_text'两个元数据字段，确保了检索的高精度）
'''
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,  # 前后3个句子作为上下文
    window_metadata_key="window", # 存储上下文 元数据的键名
    original_text_metadata_key="original_text", # 原始句子键名
)
# 将分块的句子构建节点
sentence_nodes = node_parser.get_nodes_from_documents(documents)
# 创建索引
sentence_index = VectorStoreIndex(sentence_nodes)
#
# 3.2 常规分块索引 (基准)
base_parser = SentenceSplitter(chunk_size=512)
base_nodes = base_parser.get_nodes_from_documents(documents)
base_index = VectorStoreIndex(base_nodes)

# # 4. 构建查询引擎
sentence_query_engine = sentence_index.as_query_engine(
    similarity_top_k=2, # 最相似的前两个
    # 加上上下文节点，它的作用是当查询到最相关的节点，node_postprocessors后处理器会立即从该节点的元数据中读取出上下文并且替换到原来的单个句子，传给大模型
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)
base_query_engine = base_index.as_query_engine(similarity_top_k=2
)

# 5. 执行查询并对比结果
query = "什么是LLM"
print(f"查询: {query}\n")

print("--- 句子窗口检索结果 ---")
window_response = sentence_query_engine.query(query)
print(f"回答: {window_response}\n")

print("--- 常规检索结果 ---")
base_response = base_query_engine.query(query)
print(f"回答: {base_response}\n")

查询: 什么是LLM

--- 句子窗口检索结果 ---
回答: LLM是大型语言模型（Large Language Model）的缩写，它是一种基于深度学习的人工智能模型，能够理解和生成人类语言。这类模型通常通过大量文本数据进行训练，具备语言理解、文本生成、对话交互等多种能力。

--- 常规检索结果 ---
回答: LLM是大语言模型（Large Language Model）的缩写，是一种基于深度学习的人工智能模型，能够理解和生成自然语言文本。



### 结构化索引
* 当知识库扩大到多个pdf或者文件时，查询可能与其中的一两个有关时就需要用到结构化索引来提高检索效率。
* 做法：当给文本块创建索引时，同时附加元数据（Meatdata）进行定位。
* * 查询时采用“先过滤条件，再检索”的策略

#### 先路由后检索代码示例
* 一个表格里有多个工作薄为例：
* * 先创建两个独立的向量索引：一个关于每个工作簿的相关信息的做银，组成Document对象；另一个是将每个工作簿里的数据变成一个Document对象并且附加一个元数据标签，比如年份
* * 根据查询内容先路由到要查找的工作簿，再在该工作簿中检索。

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

load_dotenv()

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 配置模型
Settings.llm = OpenAILike(
    model="deepseek-v4-pro",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv('DEEPSEEK_BASE_URL'),
    is_chat_model=True
    )
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-zh-v1.5")

# 1. 加载和预处理数据
excel_file = './movie.xlsx'
xls = pd.ExcelFile(excel_file)

summary_docs = []
content_docs = []

print("开始加载和处理Excel文件...")
for sheet_name in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet_name)

    # 数据清洗
    if '评分人数' in df.columns:
        df['评分人数'] = df['评分人数'].astype(str).str.replace('人评价', '').str.strip()
        df['评分人数'] = pd.to_numeric(df['评分人数'], errors='coerce').fillna(0).astype(int)

    # 创建摘要文档 (用于路由)
    year = sheet_name.replace('年份_', '')
    summary_text = f"这个表格包含了年份为 {year} 的电影信息，包括电影名称、导演、评分、评分人数等。"
    summary_doc = Document(
        text=summary_text,
        metadata={"sheet_name": sheet_name}
    )
    summary_docs.append(summary_doc)

    # 创建内容文档 (用于最终问答)
    content_text = df.to_string(index=False)
    content_doc = Document(
        text=content_text,
        metadata={"sheet_name": sheet_name}
    )
    content_docs.append(content_doc)

print("数据加载和处理完成。\n")

# 2. 构建向量索引
# 使用默认的内存SimpleVectorStore，它支持元数据过滤

# 2.1 为摘要创建索引
summary_index = VectorStoreIndex(summary_docs)

# 2.2 为内容创建索引
content_index = VectorStoreIndex(content_docs)

print("摘要索引和内容索引构建完成。\n")

# 3. 定义两步式查询逻辑
def query_safe_recursive(query_str):
    print(f"--- 开始执行查询 ---")
    print(f"查询: {query_str}")

    # 第一步：路由 - 在摘要索引中找到最相关的表格
    print("\n第一步：在摘要索引中进行路由...")
    summary_retriever = VectorIndexRetriever(index=summary_index, similarity_top_k=1)
    retrieved_nodes = summary_retriever.retrieve(query_str)

    if not retrieved_nodes:
        return "抱歉，未能找到相关的电影年份信息。"

    # 获取匹配到的工作表名称
    matched_sheet_name = retrieved_nodes[0].node.metadata['sheet_name']
    print(f"路由结果：匹配到工作表 -> {matched_sheet_name}")

    # 第二步：检索 - 在内容索引中根据工作表名称过滤并检索具体内容
    print("\n第二步：在内容索引中检索具体信息...")
    content_retriever = VectorIndexRetriever(
        index=content_index,
        similarity_top_k=1, # 通常只返回最匹配的整个表格即可
        filters=MetadataFilters(
            filters=[ExactMatchFilter(key="sheet_name", value=matched_sheet_name)]
        )
    )

    # 创建查询引擎并执行查询
    query_engine = RetrieverQueryEngine.from_args(content_retriever)
    response = query_engine.query(query_str)

    print("--- 查询执行结束 ---\n")
    return response

# 4. 执行查询
query = "1994年评分人数最少的电影是哪一部？"
response = query_safe_recursive(query)

print(f"最终回答: {response}")

开始加载和处理Excel文件...
数据加载和处理完成。

摘要索引和内容索引构建完成。

--- 开始执行查询 ---
查询: 1994年评分人数最少的电影是哪一部？

第一步：在摘要索引中进行路由...
路由结果：匹配到工作表 -> 年份_1994

第二步：在内容索引中检索具体信息...


PermissionDeniedError: Error code: 403 - {'error': {'code': 'unsupported_country_region_territory', 'message': 'Country, region, or territory not supported', 'param': None, 'type': 'request_forbidden'}}

# 检索优化


## 混合检索（稀疏向量加密集向量）
* 稀疏向量又叫“词法向量”通常是一个维度极高且大多数元素为零的向量，一个维度代表一个词，能进行精确检索(代表算法：BM25)
* 密集向量又叫“语义向量”通常是一个低维稠密的浮点数的向量，能够理解语义以及上下文，泛化能力强；但是得需要大量的数据训练（代表：Word2Vec以及所有基于Transformer的模型（GPT、BERT））
* **混合检索旨在同时利用稀疏向量的精确性和密集向量的泛化性，以应对复杂多变的搜索需求**


### 混合检索的两种算法：
* 倒数排序融合（RRF）
* * RRF 不关心不同检索系统的原始得分，只关心每个文档在各自结果集中的排名。其思想是：一个文档在不同检索系统中的排名越靠前，它的最终得分就越高。
* 加权线性组合
* * 将不同检索系统的得分进行归一化，再通过一个权重参数a来进行线性组合。

### 通过 Milvus 实现混合检索

#### 进行相关配置 连接Milvus 连接嵌入模型

In [4]:
import json
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import numpy as np
from pymilvus import connections, MilvusClient, FieldSchema, CollectionSchema, DataType, Collection, AnnSearchRequest, RRFRanker
from pymilvus.model.hybrid import BGEM3EmbeddingFunction

# 1. 初始化设置
COLLECTION_NAME = "dragon_hybrid_demo"
MILVUS_URI = "http://localhost:19530"  # 服务器模式
DATA_PATH = "./dragon.json"  # 相对路径
BATCH_SIZE = 50

# 2. 连接 Milvus 并初始化嵌入模型
print(f"--> 正在连接到 Milvus: {MILVUS_URI}")
connections.connect(uri=MILVUS_URI)

print("--> 正在初始化 BGE-M3 嵌入模型...")
ef = BGEM3EmbeddingFunction(use_fp16=False, device="cpu")
print(f"--> 嵌入模型初始化完成。密集向量维度: {ef.dim['dense']}")

--> 正在连接到 Milvus: http://localhost:19530
--> 正在初始化 BGE-M3 嵌入模型...


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 6658.68it/s]


--> 嵌入模型初始化完成。密集向量维度: 1024


#### 创建Collection对象，并设置字段等相关信息

In [5]:
milvus_client = MilvusClient(uri=MILVUS_URI)
if milvus_client.has_collection(COLLECTION_NAME):
    print(f"--> 正在删除已存在的 Collection '{COLLECTION_NAME}'...")
    milvus_client.drop_collection(COLLECTION_NAME)

--> 正在删除已存在的 Collection 'dragon_hybrid_demo'...


In [6]:
# 创建并设置Collection对象集合以及相关字段

'''
pk: 主键设计，auto_id=True 让 Milvus 自动生成唯一标识，避免主键冲突
标量字段: 7个VARCHAR字段用于存储元数据，max_length 根据实际数据分布优化存储
稀疏向量: SPARSE_FLOAT_VECTOR 类型，存储关键词权重
密集向量: FLOAT_VECTOR 类型，固定1024维，存储语义特征
'''
fields = [
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, auto_id=True, max_length=100),
    FieldSchema(name="img_id", dtype=DataType.VARCHAR, max_length=100),
    FieldSchema(name="path", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="description", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="location", dtype=DataType.VARCHAR, max_length=128),
    FieldSchema(name="environment", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=ef.dim["dense"])
]

# 如果集合不存在，则创建它及索引
if not milvus_client.has_collection(COLLECTION_NAME):
    print(f"--> 正在创建 Collection '{COLLECTION_NAME}'...")
    schema = CollectionSchema(fields, description="关于龙的混合检索示例")
    # 创建集合
    collection = Collection(name=COLLECTION_NAME, schema=schema, consistency_level="Strong")
    print("--> Collection 创建成功。")

    # 创建索引
    print("--> 正在为新集合创建索引...")
    sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
    collection.create_index("sparse_vector", sparse_index)
    print("稀疏向量索引创建成功。")

    dense_index = {"index_type": "AUTOINDEX", "metric_type": "IP"}
    collection.create_index("dense_vector", dense_index)
    print("密集向量索引创建成功。")

collection = Collection(COLLECTION_NAME)
collection.load()
print(f"--> Collection '{COLLECTION_NAME}' 已加载到内存。")

--> 正在创建 Collection 'dragon_hybrid_demo'...
--> Collection 创建成功。
--> 正在为新集合创建索引...
稀疏向量索引创建成功。
密集向量索引创建成功。
--> Collection 'dragon_hybrid_demo' 已加载到内存。


#### json文件的加载与预处理

In [7]:
# json数据加载与预处理
if collection.is_empty:
    print(f"--> Collection 为空，开始插入数据...")
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    docs, metadata = [], []
    for item in dataset:
        parts = [
            item.get('title', ''),
            item.get('description', ''),
            item.get('location', ''),
            item.get('environment', ''),
        ]
        docs.append(' '.join(filter(None, parts)))
        metadata.append(item)

--> Collection 为空，开始插入数据...


#### 向量生成

In [8]:

print("--> 正在生成向量嵌入...")
embeddings = ef(docs)
print("--> 向量生成完成。")

# 获取两种向量
sparse_vectors = embeddings["sparse"]    # 稀疏向量：词频统计
dense_vectors = embeddings["dense"]      # 密集向量：语义编码

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


--> 正在生成向量嵌入...
--> 向量生成完成。


#### Collection 批量插入数据

In [9]:
# 为每个字段准备批量数据 七个标量
img_ids = [doc["img_id"] for doc in metadata]
paths = [doc["path"] for doc in metadata]
titles = [doc["title"] for doc in metadata]
descriptions = [doc["description"] for doc in metadata]
categories = [doc["category"] for doc in metadata]
locations = [doc["location"] for doc in metadata]
environments = [doc["environment"] for doc in metadata]

# 插入数据
collection.insert([
    img_ids, paths, titles, descriptions, categories, locations, environments,
    sparse_vectors, dense_vectors # 两个向量
])
collection.flush() # 强制将内存缓冲区数据写入磁盘，使数据立即可搜索
collection

<Collection>:
-------------
<name>: dragon_hybrid_demo
<description>: 关于龙的混合检索示例
<schema>: {'auto_id': True, 'description': '关于龙的混合检索示例', 'fields': [{'name': 'pk', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}, 'is_primary': True, 'auto_id': True}, {'name': 'img_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'path', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 256}}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 256}}, {'name': 'description', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'name': 'category', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 64}}, {'name': 'location', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}}, {'name': 'environment', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length'

#### 创建查询条件同时生成查询向量

In [10]:
# 6. 执行搜索
search_query = "悬崖上的巨龙"
search_filter = 'category in ["western_dragon", "chinese_dragon", "movie_character"]'
top_k = 5

print(f"\n{'='*20} 开始混合搜索 {'='*20}")
print(f"查询: '{search_query}'")
print(f"过滤器: '{search_filter}'")

# 生成查询向量
query_embeddings = ef([search_query])
dense_vec = query_embeddings["dense"][0]  # 稀疏向量
sparse_vec = query_embeddings["sparse"]._getrow(0)  # 密集向量


==================== 开始混合搜索 ====================
查询: '悬崖上的巨龙'
过滤器: 'category in ["western_dragon", "chinese_dragon", "movie_character"]'


#### 三类查询  密集、稀疏、密集 + 稀疏

In [11]:
search_params = {"metric_type": "IP", "params": {}}
# 先执行单独的搜索
print("\n--- [单独] 密集向量搜索结果 ---")
dense_results = collection.search(
    [dense_vec], # 要查询的向量
    anns_field="dense_vector", # 在哪个字段名上进行相似度搜索
    param=search_params, # 搜索参数，里边有距离计算方法
    limit=top_k, # 返回结果数量
    expr=search_filter, # 过滤条件
    output_fields=["title", "path", "description", "category", "location", "environment"] # 返回的标量字段
)[0] # 返回第一个查询结果

for i, hit in enumerate(dense_results):
    print(f"{i+1}. {hit.entity.get('title')} (Score: {hit.distance:.4f})")
    print(f"    路径: {hit.entity.get('path')}")
    print(f"    描述: {hit.entity.get('description')[:100]}...")



print("\n--- [单独] 稀疏向量搜索结果 ---")
sparse_results = collection.search(
    [sparse_vec], # 稀疏向量
    anns_field="sparse_vector",
    param=search_params,
    limit=top_k,
    expr=search_filter,
    output_fields=["title", "path", "description", "category", "location", "environment"]
)[0]

for i, hit in enumerate(sparse_results):
    print(f"{i+1}. {hit.entity.get('title')} (Score: {hit.distance:.4f})")
    print(f"    路径: {hit.entity.get('path')}")
    print(f"    描述: {hit.entity.get('description')[:100]}...")

print("\n--- [混合] 稀疏+密集向量搜索结果 ---")
# 创建 RRF 融合器
rerank = RRFRanker(k=60)

# 创建搜索请求
dense_req = AnnSearchRequest([dense_vec], "dense_vector", search_params, limit=top_k)
sparse_req = AnnSearchRequest([sparse_vec], "sparse_vector", search_params, limit=top_k)

# 执行混合搜索
results = collection.hybrid_search(
    [sparse_req, dense_req],
    rerank=rerank,
    limit=top_k,
    output_fields=["title", "path", "description", "category", "location", "environment"]
)[0]

# 打印最终结果
for i, hit in enumerate(results):
    print(f"{i+1}. {hit.entity.get('title')} (Score: {hit.distance:.4f})")
    print(f"    路径: {hit.entity.get('path')}")
    print(f"    描述: {hit.entity.get('description')[:100]}...")



--- [单独] 密集向量搜索结果 ---
1. 悬崖上的白龙 (Score: 0.7214)
    路径: ../../data/C3/dragon/dragon02.png
    描述: 一头雄伟的白色巨龙栖息在悬崖边缘，背景是金色的云霞和远方的海岸。它拥有巨大的翅膀和优雅的身姿，是典型的西方奇幻生物。...
2. 中华金龙 (Score: 0.5353)
    路径: ../../data/C3/dragon/dragon06.png
    描述: 一条金色的中华龙在祥云间盘旋，它身形矫健，龙须飘逸，展现了东方神话中龙的威严与神圣。...
3. 驯龙高手：无牙仔 (Score: 0.5231)
    路径: ../../data/C3/dragon/dragon05.png
    描述: 在电影《驯龙高手》中，主角小嗝嗝骑着他的龙伙伴无牙仔在高空飞翔。他们飞向灿烂的太阳，下方是岛屿和海洋，画面充满了冒险与友谊。...

--- [单独] 稀疏向量搜索结果 ---
1. 悬崖上的白龙 (Score: 0.2254)
    路径: ../../data/C3/dragon/dragon02.png
    描述: 一头雄伟的白色巨龙栖息在悬崖边缘，背景是金色的云霞和远方的海岸。它拥有巨大的翅膀和优雅的身姿，是典型的西方奇幻生物。...
2. 中华金龙 (Score: 0.0857)
    路径: ../../data/C3/dragon/dragon06.png
    描述: 一条金色的中华龙在祥云间盘旋，它身形矫健，龙须飘逸，展现了东方神话中龙的威严与神圣。...
3. 驯龙高手：无牙仔 (Score: 0.0639)
    路径: ../../data/C3/dragon/dragon05.png
    描述: 在电影《驯龙高手》中，主角小嗝嗝骑着他的龙伙伴无牙仔在高空飞翔。他们飞向灿烂的太阳，下方是岛屿和海洋，画面充满了冒险与友谊。...

--- [混合] 稀疏+密集向量搜索结果 ---
1. 悬崖上的白龙 (Score: 0.0328)
    路径: ../../data/C3/dragon/dragon02.png
    描述: 一头雄伟的白色巨龙栖息在悬崖边缘，背景是金色的云霞和远方的海岸。它拥有巨大的翅膀和优雅的身姿，是

## 查询构建（LangChain中的SelfQueryRetriever）

### 主要的工作流程：
* 数据预处理以及创建向量存储
* 给LLM定义元数据结构（文档内容和每个元数据字段的含义及类型）
* 创建自查询检索器**SelfQueryRetriever.from_llm**：
* * 加载查询构造器：利用LLM将用户的自然语言查询转换成通用的结构化查询对象
* * 获取内置翻译器：上一步生成通用查询对象翻译成向量数据库能够原生理解和执行的语法
* 执行查询**retriever.invoke**：用自然语言发起调用。检索器内部会依次执行“构造”和“翻译”两个步骤，最终向 Chroma 发起一个同时包含语义搜索和精确元数据过滤的复合查询，从而返回最相关的结果。

## 文本查询关系数据库(用自然语言利用LLM写查询语句到MySQL中查数据)
### 该项目的业务挑战：
* 幻觉：LLM可能会想象出数据库中不存在的数据（解决方案：给他精确的数据库的结构标的相关字段）
* 对数据库结构理解不足（解决方案：在提示词中给出示例）
* 用户输入模糊或者拼写错误等表达（解决方案：如果数据库返回错误 让LLM进行反思改进）

## 查询重构以及分发
* 查询翻译：将用户的原文问题转换成一个或多个更适合检索的形式
* 查询路由：让LLM只能得分发到最合适的数据源或检索器

### 查询翻译
* 优化提示词，引导LLM输出更具体清晰的叙述。
* 多查询分解：当用户提出一个复杂的问题时，可以把它拆分成多个简单而具体的问题，各个检索 然后汇总去重然后再交给LLM生成答案。
* 假设性文档嵌入：根据用户问题先让LLM生成一个详细的文档，然后将文档转换为向量，根据这个文档在向量数据库中进行搜索，最终生成上下文。


### 查询路由
* 当系统接入了多个不同的数据源或者处理能力时，就是要一个调度台，根据语义理解自动判断最合适的处理路径

#### 基于LLM意图识别
* 就是在提示词中写清楚分类器，定义路由分支**RunnableBranch**，组合完整的路由链条


In [12]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableBranch

load_dotenv() # 加载环境变量

llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
    )


# 1. 设置不同菜系的处理链
sichuan_prompt = ChatPromptTemplate.from_template(
    "你是一位川菜大厨。请用正宗的川菜做法，回答关于「{question}」的问题。"
)
sichuan_chain = sichuan_prompt | llm | StrOutputParser()

cantonese_prompt = ChatPromptTemplate.from_template(
    "你是一位粤菜大厨。请用经典的粤菜做法，回答关于「{question}」的问题。"
)
cantonese_chain = cantonese_prompt | llm | StrOutputParser()

# 定义备用通用链
general_prompt = ChatPromptTemplate.from_template(
    "你是一个美食助手。请回答关于「{question}」的问题。"
)
general_chain = general_prompt | llm | StrOutputParser()


# 2. 创建路由链
classifier_prompt = ChatPromptTemplate.from_template(
    """根据用户问题中提到的菜品，将其分类为：['川菜', '粤菜', 或 '其他']。
    不要解释你的理由，只返回一个单词的分类结果。
    问题: {question}"""
)
#  StrOutputParser 会提取其中的 content 字段，直接返回干净的字符串。
classifier_chain = classifier_prompt | llm | StrOutputParser()

# 定义路由分支
# 类似 if-elif-else语句
router_branch = RunnableBranch(
    (lambda x: "川菜" in x["topic"], sichuan_chain),
    (lambda x: "粤菜" in x["topic"], cantonese_chain),
    general_chain  # 默认选项
)

# 组合成完整路由链
# 先根据用户问题判断出哪类菜，返回topic同时保留question,然后再将这俩传递给router_branch 决定最终的路由的决策
full_router_chain = {"topic": classifier_chain, "question": lambda x: x["question"]} | router_branch
print("完整的路由链创建成功。\n")


# 3. 运行演示查询
demo_questions = [
    {"question": "麻婆豆腐怎么做？"},      # 应该路由到川菜
    {"question": "白切鸡的正宗做法是什么？"}, # 应该路由到粤菜
    {"question": "番茄炒蛋需要放糖吗？"}      # 应该路由到其他
]


for i, item in enumerate(demo_questions, 1):
    question = item["question"]
    print(f"\n--- 问题 {i}: {question} ---")

    try:
        # 获取路由决策
        topic = classifier_chain.invoke({"question": question})
        print(f"路由决策: {topic}")

        # 执行完整链
        result = full_router_chain.invoke(item)
        print(f"回答: {result}")
    except Exception as e:
        print(f"执行错误: {e}")



完整的路由链创建成功。


--- 问题 1: 麻婆豆腐怎么做？ ---
路由决策: 川菜
回答: 要做出正宗的麻婆豆腐，讲究的是“麻、辣、烫、鲜、香、酥、嫩、整”八字真诀。作为川菜大厨，我这就把家传的方子教给你，每一步都是关键。

---

### 正宗麻婆豆腐做法

#### 一、食材准备（2-3人份）

- **主料**：嫩豆腐（南豆腐）1块（约400克），切成2厘米见方的方块。
- **牛肉末**：50克（正宗必用牛肉，比猪肉更香酥）。
- **调料**：
  - 郫县豆瓣酱（剁细）2大勺
  - 豆豉（剁细）1小勺
  - 蒜末、姜末各1小勺
  - 辣椒面（二荆条或朝天椒粉）1大勺
  - 花椒粉（现焙现磨最佳）1大勺
  - 水淀粉（玉米淀粉+水）适量
  - 盐、白糖、生抽、鸡精少许
  - 青蒜苗（切碎）2根
  - 高汤或清水200毫升

#### 二、关键步骤

1. **豆腐焯水（去腥增嫩）**  
   烧一锅水，加1小勺盐，水开后下豆腐块，小火煮2分钟，捞出沥干。这一步让豆腐更紧实不易碎，同时去除豆腥味。

2. **炒底料（出红油）**  
   热锅冷油（菜籽油最佳），下牛肉末中火煸炒至酥香（微微焦黄）。转小火，加豆瓣酱、豆豉、姜蒜末，慢慢炒出红油（约1分钟）。撒入辣椒面，快速翻炒出香（注意别炒糊）。

3. **烧豆腐（入味）**  
   倒入高汤或清水，大火烧开。轻轻滑入豆腐，加少许生抽、白糖（提鲜）、鸡精。转中小火，让豆腐在汤汁中咕嘟3-5分钟，期间不要用锅铲乱翻，可以轻晃锅子防粘。

4. **勾芡（三次收汁）**  
   分三次淋入水淀粉：第一次薄芡，让汤汁变稠；第二次中芡，包裹豆腐；第三次厚芡，让芡汁紧贴豆腐表面。每次勾芡后轻推豆腐，让芡汁均匀。

5. **点睛之笔**  
   出锅前撒入青蒜碎，翻匀。装盘后，趁热撒上现磨的花椒粉（一定要最后撒，否则麻味挥发）。

#### 三、大厨的“八字诀”解析

- **麻**：汉源花椒现焙现磨，麻味醇正不苦。
- **辣**：郫县豆瓣+辣椒面，辣而不燥。
- **烫**：出锅即上桌，豆腐内部滚烫。
- **鲜**：高汤提鲜，牛肉末增香。
- **香**：豆豉、蒜、青蒜复合香气。
- **酥**：牛肉末煸至酥脆，嚼之有颗粒感。
- **嫩**：豆腐焯水+小火慢烧，入口即化。


#### 嵌入相似性路由
* 不用依赖LLM分类，延迟低，通过计算用户查询与预设的路由实例语句之间的向量嵌入相似度来做出决策
* 工作流程：定义路由描述并转化为向量；
* * 创建路由要分发的目标处理链并创建字典coute_map 将路由名称和链对应起来；
* * 定义一个路由函数，将接收用户问题，并且计算与个路由描述的相似度，并且选择最相似的路由并调用处理链
* * 合并路由函数包装成RunnableLambda函数，形成完成的路由链

In [13]:
##### 代码实例：
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnablePassthrough
from langchain_community.utils.math import cosine_similarity
import numpy as np


# 1. 定义路由描述
sichuan_route_prompt = "你是一位处理川菜的专家。用户的问题是关于麻辣、辛香、重口味的菜肴，例如水煮鱼、麻婆豆腐、鱼香肉丝、宫保鸡丁、花椒、海椒等。"
cantonese_route_prompt = "你是一位处理粤菜的专家。用户的问题是关于清淡、鲜美、原汁原味的菜肴，例如白切鸡、老火靓汤、虾饺、云吞面等。"

route_prompts = [sichuan_route_prompt, cantonese_route_prompt]
route_names = ["川菜", "粤菜"]

# 初始化嵌入模型，并对路由描述进行向量化
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")
route_prompt_embeddings = embeddings.embed_documents(route_prompts)
print(f"已定义 {len(route_names)} 个路由: {', '.join(route_names)}")

# 2. 定义不同路由的目标链
llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
    )

# 定义川菜和粤菜处理链
sichuan_chain = (
    PromptTemplate.from_template("你是一位川菜大厨。请用正宗的川菜做法，回答关于「{query}」的问题。")
    | llm
    | StrOutputParser()
)
cantonese_chain = (
    PromptTemplate.from_template("你是一位粤菜大厨。请用经典的粤菜做法，回答关于「{query}」的问题。")
    | llm
    | StrOutputParser()
)

route_map = { "川菜": sichuan_chain, "粤菜": cantonese_chain }
print("川菜和粤菜的处理链创建成功。\n")

# 3. 创建路由函数
def route(info):
    # 对用户查询进行嵌入
    query_embedding = embeddings.embed_query(info["query"])

    # 计算与各路由提示的余弦相似度
    similarity_scores = cosine_similarity([query_embedding], route_prompt_embeddings)[0]

    # 找到最相似的路由
    chosen_route_index = np.argmax(similarity_scores)
    chosen_route_name = route_names[chosen_route_index]

    print(f"路由决策: 检测到问题与“{chosen_route_name}”最相似。")

    # 获取对应的处理链
    chosen_chain = route_map[chosen_route_name]

    # 直接调用选中的链并返回结果
    return chosen_chain.invoke(info)

# 创建完整的路由链
full_chain = RunnableLambda(route)


# 4. 运行演示查询
demo_queries = [
    "水煮鱼怎么做才嫩？",        # 应该路由到川菜
    "如何做一碗清淡的云吞面？",    # 应该路由到粤菜
    "麻婆豆腐的核心调料是什么？",  # 应该路由到川菜
]

for i, query in enumerate(demo_queries, 1):
    print(f"\n--- 问题 {i}: {query} ---")
    try:
        # 传入字典，full_chain 会直接返回最终答案
        result = full_chain.invoke({"query": query})
        print(f"回答: {result}")
    except Exception as e:
        print(f"执行错误: {e}")

已定义 2 个路由: 川菜, 粤菜
川菜和粤菜的处理链创建成功。


--- 问题 1: 水煮鱼怎么做才嫩？ ---
路由决策: 检测到问题与“川菜”最相似。
回答: 要做出正宗且嫩滑的水煮鱼，关键在于选料、刀工、腌制和火候。作为川菜师傅，我分享几个核心步骤：

1. **选鱼**：首选草鱼或黑鱼，肉质紧实且刺少。鱼要现杀，保证新鲜。

2. **片鱼**：鱼片需薄而均匀（约2-3毫米），顺着鱼肉的纹理斜刀片，避免切断纤维。片好后用清水漂洗去血水，沥干。

3. **腌制**（嫩滑的关键）：  
   - 加少许盐、料酒、白胡椒粉抓匀至发黏。  
   - 打入一个蛋清，继续抓匀，让蛋清裹住鱼片。  
   - 加一勺红薯淀粉（或玉米淀粉），抓匀后淋少许食用油锁住水分，静置10分钟。

4. **煮鱼**：  
   - 锅中炒香豆瓣酱、姜蒜、干辣椒、花椒，加高汤或水烧开，先煮配菜（豆芽、莴笋等）捞出垫底。  
   - 转小火，将鱼片一片片滑入汤中，不要搅动，待鱼片变白（约30秒）立即关火。  
   - 连汤带鱼倒入碗中，撒上蒜末、干辣椒段、花椒粒，淋一勺滚烫的菜籽油激香。

**关键提醒**：  
- 鱼片下锅后**切忌大火猛煮**，否则肉质变老。  
- 汤底要宽（水量足），鱼片入锅后温度下降快，能保持嫩度。  
- 最后淋油时油温要够高（冒青烟），才能逼出香料香气。

这样做出的水煮鱼，鱼片嫩滑如豆腐，麻辣鲜香，入口即化。配一碗白米饭，巴适得很！

--- 问题 2: 如何做一碗清淡的云吞面？ ---
路由决策: 检测到问题与“粤菜”最相似。
回答: 作为一位粤菜大厨，我深知一碗上好的云吞面，讲究的是“汤清、味鲜、面爽、云吞靓”。所谓“清淡”，并非寡淡无味，而是汤底清澈鲜甜，不油不腻，能尝出食材本真。下面我按经典广府做法，一步步教你做这碗云吞面。

---

### 一、汤底：清而不浊，鲜自骨来

**关键**：不用浓汤宝，不用大骨熬到发白，而是用“地鱼”（比目鱼干）和虾籽提鲜，汤色如茶。

**材料**：
- 大地鱼干（比目鱼干）1条（约手掌大小）
- 猪筒骨或鸡壳（鸡骨架）500克
- 虾籽（干货）1汤匙
- 生姜3片
- 清水3升
- 盐、白胡椒粉少许

**做法**：
1. 大地鱼干用小火烘烤至两面微焦出香（约3分钟），撕成小块。
2. 猪筒骨或鸡

#### 基于Llamalndex的查询路由功能
* Llamalndex 主要是包不同数据源或者查询策略包装成Tools（工具），然后依靠LL的意图识别来理解各个工具的语义，然后根据用户的问题来选择相应的工具。

### 检索进阶

#### 重排序

* RRF（倒数排序融合）
* * 零样本的排序方法，然后通过不同检索器来计算排名的倒数给文档打分(前几项比最后吉祥要重要的多)
* RankLLM：给大模型写提示词，让它给重排
* Cross-Encoder（交叉编码器）：
* * 首先从知识库中取出初始的文档列表，
* * 然后对每一篇文档与用户的查询进行配对并发送给Cross-Encoder模型
* * 模型对每个“查询-文档”对进行一次完整的、独立的推理计算并得出一个精确的相关性分数。
* * 系统根据这些新的分数对文档列表进行重新排序，并将最终结果返回给用户
* * 特点：**高延时、高精度**
* ColBERT重排：
* * 在Cross-Encoder重排的高精度和高效率之间得到了平衡
* * 先给查询和文档中的每个Token生成嵌入向量
* * 在查询时模型会计算查询中每个token跟文档的每个token的最大相似度 （**后期交互**机制）
* * 将查询中所有 Token 得到的最大相似度分数相加，得到最终的相关性总分。


#### 利用LangChain中的相关方法自制ColBERT重排，以及压缩


In [14]:
# 空格

#### 压缩
* 压缩其实就是对检索到的内容进行“压缩”和“提炼”，只保留用户查询最直接的相关信息。
* 内容提取。文档过滤

#### 校正
* 如果使用一些过期或者有毒的知识库来进行投喂给LLM，会出现幻觉以及非常错误的回答。
* 解决方案：引入‘自我反思’循环，让RAG先进行‘**检索->评估->行动**’
* 流程：1、检索：先从知识库中检索一组文档
* * 2、评估：然后通过一个‘检索评估器’进行判断文档与查询的相关性并且给出“正确”、“不正确”、“模型”的标签
* * 3、如果是“正确”，就会将文档分块过滤、重新组合喂给LLM
* * * 若“不正确”，会触发“知识搜索 (Knowledge Searching)”，然后对原始查询进行查询重写，生成一个更适合web搜索的查询，在进行web搜索，用外部信息来回答问题
* * * 若“模糊”：同样会触发“知识搜索”，但通常会直接使用原始查询进行 Web 搜索，以获取额外信息来辅助生成答案

## 格式化生成
* 有些情况我们需要模型输出具有特定结构的数据，如json等
* 可以用LangChain中的框架中的OutputParsers来定义格式
* 当然也可以用提示词

In [17]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import os
from dotenv import load_dotenv

llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
    )

# 1. 定义期望的数据结构
class PersonInfo(BaseModel):
    """用于存储个人信息的数据结构。"""
    name: str = Field(description="人物姓名")
    age: int = Field(description="人物年龄")
    skills: List[str] = Field(description="技能列表")

# 2. 基于 Pydantic 模型，创建解析器
parser = PydanticOutputParser(pydantic_object=PersonInfo)

# 3. 创建提示模板，注入格式指令
prompt = PromptTemplate(
    template="请根据以下文本提取信息。\n{format_instructions}\n{text}\n",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 4. 创建处理链 (假定 llm 已被初始化)
chain = prompt | llm | parser

# 5. 执行调用
text = "张三今年30岁，他擅长Python和Go语言牛逼。"
result = chain.invoke({"text": text})

# 6. 打印结果
print(result)

name='张三' age=30 skills=['Python', 'Go语言']


## Function Calling
* 是一个多轮对话流程，让模型与外部工具协同工作
* 基本的工作流程：
* * 先定义工具tools：名称、功能描述、需要的参数等
* * 用户提问：问一个需要调用工具才能回答的请求
* * 模型决策：模型会分析用户意图并匹配合适的工具，返回一个包含 tool_calls 的特殊响应
* * 代码执行：应用接收到这个指令，解析出工具名称和参数，然后在代码层面实际执行这个工具(例如调api)
* * 结果反馈：将执行结果包装陈隔一个role为tool的消息，再次发给LLM
* * 最终生成：LLM接收到执行结果后，结合问题生成自然语言返回
* 优点：LLM原生支持的能力更稳定精确、并且能够意图识别、可以与外部世界交互(调api等)

In [18]:
from openai import OpenAI
import os
import requests
from dotenv import load_dotenv


load_dotenv()

# 初始化 OpenAI 客户端
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

# 定义一个函数，用于发送消息并获取模型的响应
def send_messages(messages, tools=None):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools,
        tool_choice="auto",  # 让模型自主决定是否调用工具
    )
    return response.choices[0].message
# 调用高德api获取天气
def get_weather(location: str) -> dict:
    """
    查询实时天气

    Args:
        city: 城市名称或 adcode

    Returns:
        天气信息字典
    """
    api_key = os.getenv("GAODE_KEY")
    url = "https://restapi.amap.com/v3/weather/weatherInfo?parameters"

    params = {
        "key": api_key,
        "city": location,
        "extensions": "base",  # base: 实况天气, all: 预报天气
        "output": "json"
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()
        result = response.json()

        if result["status"] == "1" and result["lives"]:
            weather_info = result["lives"][0]
            return {
                "city": weather_info["city"],
                "weather": weather_info["weather"],
                "temperature": weather_info["temperature"],
                "wind_direction": weather_info["winddirection"],
                "wind_power": weather_info["windpower"],
                "humidity": weather_info["humidity"]
            }
        else:
            return None
    except Exception as e:
        print(f"天气查询失败: {e}")
        return None
# 1. 定义工具（函数）的 Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定地点的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市",
                    }
                },
                "required": ["location"]
            },
        }
    },
]

# 1. 用户提问，模型决策调用工具
messages = [{"role": "user", "content": "南京天气咋样"}]
print(f"User> {messages[0]['content']}\n")
message = send_messages(messages, tools=tools)
print(message)
# 2. 执行工具，并将结果返回模型
if message.tool_calls:
    print("--- 模型发起了工具调用 ---")
    tool_call = message.tool_calls[0]
    function_info = tool_call.function
    print(f"工具名称: {function_info.name}")
    print(f"工具参数: {function_info.arguments}")

    # 将模型的回复（包含工具调用请求）添加到消息历史中
    messages.append(message)

    import json
    args = json.loads(function_info.arguments)
    # 模拟执行工具
    weth = function_info.arguments
    tool_output = get_weather(args.get('location',weth))
    print(f"--- 执行工具并返回结果 ---")
    print(f"工具执行结果: {tool_output}\n")

    # 将工具的执行结果作为一个新的消息添加到历史中
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(tool_output, ensure_ascii=False)})

    # 3. 第二次调用：将工具结果返回给模型，获取最终回答
    print("--- 将工具结果返回给模型，获取最终答案 ---")
    final_message = send_messages(messages, tools=tools)
    print(f"Model> {final_message.content}")
else:
    # 如果模型没有调用工具，直接打印其回答
    print(f"Model> {message.content}")

User> 南京天气咋样

ChatCompletionMessage(content='好的，我来查一下南京的天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_R8qAh38FId0EBIL3tdfB3087', function=Function(arguments='{"location": "南京"}', name='get_weather'), type='function', index=0)])
--- 模型发起了工具调用 ---
工具名称: get_weather
工具参数: {"location": "南京"}
--- 执行工具并返回结果 ---
工具执行结果: {'city': '南京市', 'weather': '阴', 'temperature': '25', 'wind_direction': '东南', 'wind_power': '4', 'humidity': '66'}

--- 将工具结果返回给模型，获取最终答案 ---
Model> 南京当前的天气情况如下：

- 🌤 **天气**：阴天
- 🌡 **温度**：25°C
- 💨 **风向**：东南风
- 💨 **风力**：4级
- 💧 **湿度**：66%

整体来看，南京现在是阴天，气温比较舒适，不过风力稍大，出门可以带件外套～


# RAG系统评估
* 三元组：
* 上下文相关性、
* 可信度(忠实度)、
* 答案相关性。

## 分为两个部分：检索评估、响应评估
* 检索评估：对应上下文评估，主要借鉴多项指标：
* * 上下文精确率、（计算在检索到的前 k 个文档中相关文档所占的比例）
* * 上下文召回率、（计算在检索到的前 k 个文档中，找到的相关文档占所有真实相关文档总数的比例。高召回率意味着系统能够成功找回大部分关键信息）
* * F1分数、（精确率和召回率的调和平均数）
* * 平均倒数排名、（评估系统将第一个相关文档排在靠前位置的能力，适用于只只关心第一个正确答案的场景）
* * 平均准确率均值MAP，（综合性指标，同时评估了检索结果的精确率和相关文档的排名）
* 响应评估：对应忠实度和答案相关度。
* * 通过大模型进行评估，
* * 基于词汇重叠的重要指标，这种评估需要在数据集有标准答案，然后计算生成答案与标准答案之间的重叠成来评估质量。






# 两个评估工具
* [phoenix 文档](https://arize.com/docs/phoenix "点击查看phoenix详细文档")
* [Ragas 文档](https://docs.ragas.io/en/stable/getstarted/ "点击查看Ragas详细文档")